# Feature Engineering

## Imports

In [1]:
import pandas as pd

## Importar data

In [2]:
CLEAN_STORE_ROUTE = r"../data/clean/store_clean.csv"
CLEAN_TRAIN_ROUTE = r"../data/clean/train_clean.csv"

In [3]:
df_store = pd.read_csv(CLEAN_STORE_ROUTE)
df_train = pd.read_csv(CLEAN_TRAIN_ROUTE, dtype={"StateHoliday": str}) # Esto solo porque pandas vuelve a leer el csv e "infiere datos"

## Merge

In [4]:
df = df_train.merge(df_store, on="Store", how="left")

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   Store                      1017209 non-null  int64  
 1   DayOfWeek                  1017209 non-null  int64  
 2   Date                       1017209 non-null  str    
 3   Sales                      1017209 non-null  int64  
 4   Customers                  1017209 non-null  int64  
 5   Open                       1017209 non-null  int64  
 6   Promo                      1017209 non-null  int64  
 7   StateHoliday               1017209 non-null  str    
 8   SchoolHoliday              1017209 non-null  int64  
 9   StoreType                  1014567 non-null  str    
 10  Assortment                 1014567 non-null  str    
 11  CompetitionDistance        1014567 non-null  float64
 12  CompetitionOpenSinceMonth  693861 non-null   float64
 13  CompetitionOpenSinceYea

In [6]:
df["StateHoliday"].unique()

<StringArray>
['0', 'a', 'b', 'c']
Length: 4, dtype: str

## Preprocessing

### Nulos por store's en train pero no en store

Aparecieron nulos, esto porque hay tiendas que existen en train, pero no en store, eliminar tales registros

In [7]:
df = df.dropna(subset="StoreType")

In [8]:
df.info()

<class 'pandas.DataFrame'>
Index: 1014567 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   Store                      1014567 non-null  int64  
 1   DayOfWeek                  1014567 non-null  int64  
 2   Date                       1014567 non-null  str    
 3   Sales                      1014567 non-null  int64  
 4   Customers                  1014567 non-null  int64  
 5   Open                       1014567 non-null  int64  
 6   Promo                      1014567 non-null  int64  
 7   StateHoliday               1014567 non-null  str    
 8   SchoolHoliday              1014567 non-null  int64  
 9   StoreType                  1014567 non-null  str    
 10  Assortment                 1014567 non-null  str    
 11  CompetitionDistance        1014567 non-null  float64
 12  CompetitionOpenSinceMonth  693861 non-null   float64
 13  CompetitionOpenSinceYear   6

### Conversión de feature "Date" a datetime

In [9]:
df["Date"] = pd.to_datetime(df["Date"])

In [10]:
df.info()

<class 'pandas.DataFrame'>
Index: 1014567 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1014567 non-null  int64         
 1   DayOfWeek                  1014567 non-null  int64         
 2   Date                       1014567 non-null  datetime64[us]
 3   Sales                      1014567 non-null  int64         
 4   Customers                  1014567 non-null  int64         
 5   Open                       1014567 non-null  int64         
 6   Promo                      1014567 non-null  int64         
 7   StateHoliday               1014567 non-null  str           
 8   SchoolHoliday              1014567 non-null  int64         
 9   StoreType                  1014567 non-null  str           
 10  Assortment                 1014567 non-null  str           
 11  CompetitionDistance        1014567 non-null  float64 

In [11]:
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0.0,0.0,0.0,NoPromo
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1.0,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1.0,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0.0,0.0,0.0,NoPromo
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0.0,0.0,0.0,NoPromo


### Nulos en CompetitionOpenSinceMonth/Year

Ahora faltan los registros de CompetitionOpenSinceMonth/Year

Para ello definiremos hasta cuando consideraremos parte de entrenamiento

Primero ordenamos las fechas

In [12]:
df = df.sort_values("Date")

Ahora vemos desde donde parte hasta donde llega

In [13]:
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
1017208,1115,2,2013-01-01,0,0,0,0,a,1,d,c,5350.0,NaN,NaN,1.0,22.0,2012.0,"Mar,Jun,Sept,Dec"
1016473,379,2,2013-01-01,0,0,0,0,a,1,d,a,6630.0,NaN,NaN,0.0,0.0,0.0,NoPromo
1016472,378,2,2013-01-01,0,0,0,0,a,1,a,c,2140.0,8.0,2012.0,0.0,0.0,0.0,NoPromo
1016471,377,2,2013-01-01,0,0,0,0,a,1,a,c,100.0,6.0,2010.0,1.0,18.0,2010.0,"Feb,May,Aug,Nov"
1016470,376,2,2013-01-01,0,0,0,0,a,1,a,a,160.0,8.0,2012.0,0.0,0.0,0.0,NoPromo


In [14]:
df.tail()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
745,746,5,2015-07-31,9082,638,1,1,0,1,d,c,4330.0,2.0,2011.0,1.0,35.0,2011.0,"Mar,Jun,Sept,Dec"
746,747,5,2015-07-31,10708,826,1,1,0,1,c,c,45740.0,8.0,2008.0,0.0,0.0,0.0,NoPromo
747,748,5,2015-07-31,7481,578,1,1,0,1,d,a,2380.0,3.0,2010.0,1.0,14.0,2011.0,"Jan,Apr,Jul,Oct"
741,742,5,2015-07-31,10460,1016,1,1,0,1,d,c,4380.0,NaN,NaN,0.0,0.0,0.0,NoPromo
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0.0,0.0,0.0,NoPromo


Llega hasta mas o menos mitades del 2015, asi que ocuparemos todo 2013 y todo 2014 como entrenamiento

Por lo que creamos un train temporal para calcular su mediana en CompetitionOpenSinceMonth y mediana en CompetitionOpenSinceYear

In [15]:
temporal_df_train = df[
    (df["Date"] < "2015-01-01")
]

In [16]:
median_competition_open_since_month = temporal_df_train["CompetitionOpenSinceMonth"].median()
median_competition_open_since_year = temporal_df_train["CompetitionOpenSinceYear"].median()

In [17]:
df["CompetitionOpenSinceMonth"] = df["CompetitionOpenSinceMonth"].fillna(median_competition_open_since_month)
df["CompetitionOpenSinceYear"] = df["CompetitionOpenSinceYear"].fillna(median_competition_open_since_year)

In [18]:
df.info()

<class 'pandas.DataFrame'>
Index: 1014567 entries, 1017208 to 0
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1014567 non-null  int64         
 1   DayOfWeek                  1014567 non-null  int64         
 2   Date                       1014567 non-null  datetime64[us]
 3   Sales                      1014567 non-null  int64         
 4   Customers                  1014567 non-null  int64         
 5   Open                       1014567 non-null  int64         
 6   Promo                      1014567 non-null  int64         
 7   StateHoliday               1014567 non-null  str           
 8   SchoolHoliday              1014567 non-null  int64         
 9   StoreType                  1014567 non-null  str           
 10  Assortment                 1014567 non-null  str           
 11  CompetitionDistance        1014567 non-null  float64 

### Eliminar registros donde las ventas son 0

Es necesario hacer esto porque en la evaluación no podemos dividir po 0

Además, podemos hacer esto porque los sales en 0 en su mayoría son días que no se abre la tienda, entonces evidentemente las ventas son 0, también veremos si son muchos registros con sales en 0

In [19]:
print(f"Cantidad de registros en 0: {len(df[df['Sales'] == 0])}")
print(f"Cantidad de registros en 0 siendo Open=0: {len(df[(df['Sales'] == 0) & (df['Open'] == 0)])}")
print(f"Cantidad de registros totales: {len(df)}")

Cantidad de registros en 0: 172415
Cantidad de registros en 0 siendo Open=0: 172361
Cantidad de registros totales: 1014567


In [20]:
df = df[
    (df["Sales"] != 0)
]

## Reset index

Esto es necesario para que quedemos con un dataframe ordenado por fecha con indices ordenados

In [21]:
df = df.reset_index(drop=True)

## Feature engineering

### Crear features de tiempo con Date

Esto es necesario ya que los modelos no pueden interpretar una fecha

In [22]:
df["Day"] = df["Date"].dt.day

In [23]:
df["Month"] = df["Date"].dt.month

In [24]:
df["Year"] = df["Date"].dt.year

In [25]:
df["Week"] = df["Date"].dt.isocalendar().week

In [26]:
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Day,Month,Year,Week
0,353,2,2013-01-01,3139,820,1,0,a,1,b,...,8.0,2010.0,1.0,14.0,2013.0,"Feb,May,Aug,Nov",1,1,2013,1
1,335,2,2013-01-01,2401,482,1,0,a,1,b,...,8.0,2010.0,1.0,31.0,2013.0,"Jan,Apr,Jul,Oct",1,1,2013,1
2,512,2,2013-01-01,2646,625,1,0,a,1,b,...,8.0,2010.0,1.0,5.0,2013.0,"Mar,Jun,Sept,Dec",1,1,2013,1
3,494,2,2013-01-01,3113,527,1,0,a,1,b,...,6.0,2011.0,0.0,0.0,0.0,NoPromo,1,1,2013,1
4,530,2,2013-01-01,2907,532,1,0,a,1,a,...,8.0,2010.0,0.0,0.0,0.0,NoPromo,1,1,2013,1


### Eliminar Customers

Aunque sea un buen indicador de las ventas, no podemos saber cuantos customers van a haber

In [27]:
df = df.drop(columns=["Customers"])

### Crear Lag's

Esto va a representar las ventas de hace 1 día, hace 7 días, 14 y 30

Primero es necesario asegurar que tenemos ordenados store y date, es decir la store 1 con fecha 2013-01-01, luego store 1 con fecha 2013-01-02 y así sucesivamente. Esto con el fin de poder luego hacer shift en el grupo de la store 1 y tener todas las fechas ordenadas, luego corrererlas 1 espacio, 7, 14 y 30

In [28]:
df = df.sort_values(by=["Store", "Date"]).reset_index(drop=True)

Ahora si hacemos grupos en cada store y corremos el shift

In [29]:
lags = [1, 7, 14, 30]

In [30]:
for lag in lags:
    df[f"lag_{lag}"] = df.groupby("Store")["Sales"].shift(lag)

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 842152 entries, 0 to 842151
Data columns (total 25 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Store                      842152 non-null  int64         
 1   DayOfWeek                  842152 non-null  int64         
 2   Date                       842152 non-null  datetime64[us]
 3   Sales                      842152 non-null  int64         
 4   Open                       842152 non-null  int64         
 5   Promo                      842152 non-null  int64         
 6   StateHoliday               842152 non-null  str           
 7   SchoolHoliday              842152 non-null  int64         
 8   StoreType                  842152 non-null  str           
 9   Assortment                 842152 non-null  str           
 10  CompetitionDistance        842152 non-null  float64       
 11  CompetitionOpenSinceMonth  842152 non-null  float64       
 12 

Hay que eliminar los nulos generados

In [32]:
df = df.dropna()

In [33]:
df.info()

<class 'pandas.DataFrame'>
Index: 808792 entries, 30 to 842151
Data columns (total 25 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Store                      808792 non-null  int64         
 1   DayOfWeek                  808792 non-null  int64         
 2   Date                       808792 non-null  datetime64[us]
 3   Sales                      808792 non-null  int64         
 4   Open                       808792 non-null  int64         
 5   Promo                      808792 non-null  int64         
 6   StateHoliday               808792 non-null  str           
 7   SchoolHoliday              808792 non-null  int64         
 8   StoreType                  808792 non-null  str           
 9   Assortment                 808792 non-null  str           
 10  CompetitionDistance        808792 non-null  float64       
 11  CompetitionOpenSinceMonth  808792 non-null  float64       
 12  Com

### Rolling mean

Ahora trabajaremos el promedio de los últimos 7, 14 y 30 días

Corroboramos que tenemos el dataframe ordenado

In [34]:
df.head()

,Store,DayOfWeek,Date,Sales,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,...,Promo2SinceYear,PromoInterval,Day,Month,Year,Week,lag_1,lag_7,lag_14,lag_30
30,1,3,2013-02-06,6140,1,1,0,0,c,a,...,0.0,NoPromo,6,2,2013,6,6049.0,3725.0,5394.0,5530.0
31,1,4,2013-02-07,5499,1,1,0,0,c,a,...,0.0,NoPromo,7,2,2013,6,6140.0,4601.0,5720.0,4327.0
32,1,5,2013-02-08,5681,1,1,0,0,c,a,...,0.0,NoPromo,8,2,2013,6,5499.0,4709.0,5578.0,4486.0
33,1,6,2013-02-09,5370,1,0,0,0,c,a,...,0.0,NoPromo,9,2,2013,6,5681.0,5633.0,5195.0,4997.0
34,1,1,2013-02-11,4409,1,0,0,0,c,a,...,0.0,NoPromo,11,2,2013,7,5370.0,5970.0,5586.0,7176.0


Ahora creamos el rolling mean

In [35]:
rolling_days = [7, 14, 30]

Obtenemos toda la Serie de Sales del grupo, la transformamos donde x es la serie completa, se le corre 1 espacio hacia "abajo" (para no utilizar el dato del mismo dia dentro del promedio, ocupa los 3 anteriores) y luego abre una ventan de "7 dias" en el primer caso, va al primer registro, 7 días hacia atrás, obvio no tiene, luego al segundo, tampoco tiene 7 días para atras, hasta que llega con el que si tiene 7 días, considerandose a si mismo como dia anterior 1, luego con 2 días atrás hasta 7, de esos días saca el promedio y lo coloca en la nueva columna

In [36]:
for rolling in rolling_days:
    df[f"rolling_mean_{rolling}"] = df.groupby("Store")["Sales"].transform(
        lambda x: x.shift(1).rolling(rolling).mean()
    )

In [37]:
df.info()

<class 'pandas.DataFrame'>
Index: 808792 entries, 30 to 842151
Data columns (total 28 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Store                      808792 non-null  int64         
 1   DayOfWeek                  808792 non-null  int64         
 2   Date                       808792 non-null  datetime64[us]
 3   Sales                      808792 non-null  int64         
 4   Open                       808792 non-null  int64         
 5   Promo                      808792 non-null  int64         
 6   StateHoliday               808792 non-null  str           
 7   SchoolHoliday              808792 non-null  int64         
 8   StoreType                  808792 non-null  str           
 9   Assortment                 808792 non-null  str           
 10  CompetitionDistance        808792 non-null  float64       
 11  CompetitionOpenSinceMonth  808792 non-null  float64       
 12  Com

Quitar nulos generados

In [38]:
df = df.dropna().reset_index(drop=True)

### Desviación estandar de ventas anteriores

Aquí lo que haremos es calcular la volatilidad de las ventas anteriores

In [39]:
df = df.sort_values(by=["Store", "Date"])

In [40]:
df.head()

,Store,DayOfWeek,Date,Sales,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,...,Month,Year,Week,lag_1,lag_7,lag_14,lag_30,rolling_mean_7,rolling_mean_14,rolling_mean_30
0,1,3,2013-03-13,4341,1,0,0,0,c,a,...,3,2013,11,3853.0,6300.0,4038.0,6140.0,5449.000000,5203.214286,5212.200000
1,1,4,2013-03-14,5108,1,0,0,0,c,a,...,3,2013,11,4341.0,5973.0,3794.0,5499.0,5169.142857,5224.857143,5152.233333
2,1,5,2013-03-15,4925,1,0,0,0,c,a,...,3,2013,11,5108.0,5637.0,4558.0,5681.0,5045.571429,5318.714286,5139.200000
3,1,6,2013-03-16,5003,1,0,0,0,c,a,...,3,2013,11,4925.0,5853.0,4676.0,5370.0,4943.857143,5344.928571,5114.000000
4,1,1,2013-03-18,7072,1,1,0,0,c,a,...,3,2013,12,5003.0,5578.0,4611.0,4409.0,4822.428571,5368.285714,5101.766667


In [41]:
rolling_days_std = [3, 7, 14, 30]

In [42]:
for roll_std in rolling_days_std:
    df[f"rolling_std_{roll_std}"] = df.groupby("Store")["Sales"].transform(
        lambda x: x.shift(1).rolling(roll_std).std()
    )

In [43]:
df = df.dropna().reset_index(drop=True)

## Guardar en processed

Este es el dataframe final que se utilizará para Machine Learning

In [44]:
df.to_csv("../data/processed/df_processed.csv", index=False)